Histograms and Scatter Plots to Analyze Clustering Betweeen Particle Types

In [14]:
# imports
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import h5py

import matplotlib.pyplot as plt
import seaborn as sns

import re
# env confirmation
print("Python executable:", sys.executable)

# plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

print("Setup complete.")


Python executable: /Users/k92/clustering_env/bin/python
Setup complete.


In [15]:
#verify imports real quick
import numpy
import pandas
import h5py
import matplotlib
import seaborn
from pathlib import Path

print("All imports successful.")


All imports successful.


In [16]:
# set where hdf5 files live in order to loop through for plotting
# assumes they live within this repo as ./metrics/
# filename convention:
# ./metrics/metrics_<particle>_<energy><unit>_<optional_suffix>.h5

metrics_dir = Path("./metrics")

FNAME_RE = re.compile(
    r"^metrics_(?P<particle>.+)_(?P<energy>\d+)(?P<unit>keV|MeV|GeV)(?:_(?P<suffix>.*))?\.h5$",
    re.IGNORECASE
)

def parse_run_name(fname: str):
    """
    Parses filenames like:
      metrics_alpha_500keV_1000cs.h5

    Returns:
      particle, energy_keV, energy_text, suffix, run_id
    """
    m = FNAME_RE.match(fname)
    if not m:
        return None

    particle = m.group("particle")
    energy_num = int(m.group("energy"))
    unit_original = m.group("unit")
    unit = unit_original.lower()
    suffix = m.group("suffix")

    energy_keV = float(energy_num)
    if unit == "mev":
        energy_keV *= 1e3
    elif unit == "gev":
        energy_keV *= 1e6

    energy_text = f"{energy_num}{unit_original}"

    if suffix:
        run_id = f"{particle}_{energy_text}_{suffix}"
    else:
        run_id = f"{particle}_{energy_text}"

    return {
        "particle": particle,
        "energy_keV": energy_keV,
        "energy_text": energy_text,
        "suffix": suffix,
        "run_id": run_id,
    }

print("Metrics folder:", metrics_dir.resolve())
print("Expected naming: metrics_<particle>_<energy><unit>_<optional_suffix>.h5")

Metrics folder: /Users/k92/Allpix2ClusteringExample/metrics
Expected naming: metrics_<particle>_<energy><unit>_<optional_suffix>.h5


In [17]:
UNIT_TO_KEV = {"kev": 1, "mev": 1_000, "gev": 1_000_000}

def parse_energy_to_keV(text: str):
    """
    Parse strings like: '10keV', '1MeV', '2GeV' -> energy_keV (int)
    Returns (energy_keV, unit) or (None, None) if not parseable.
    """
    m = re.match(r"^\s*(\d+)\s*(keV|MeV|GeV)\s*$", text, re.IGNORECASE)
    if not m:
        return None, None
    val = int(m.group(1))
    unit = m.group(2).lower()
    return val * UNIT_TO_KEV[unit], unit

def parse_run_from_folder(folder_name: str):
    """
    returns dict with particle, energy_keV, energy_text, suffix, run_id
    Accepts:
      gamma_10keV
      gamma_10keV_tcad
    """
    m = re.match(
        r"^(?P<particle>.+)_(?P<energy>\d+(?:keV|MeV|GeV))(?:_(?P<suffix>.*))?$",
        folder_name,
        re.IGNORECASE
    )
    if not m:
        return None

    particle = m.group("particle")
    energy_text = m.group("energy")
    suffix = m.group("suffix")  # will be None if not present

    energy_keV, unit = parse_energy_to_keV(energy_text)
    if energy_keV is None:
        return None

    if suffix:
        run_id = f"{particle}_{energy_text}_{suffix}"
    else:
        run_id = f"{particle}_{energy_text}"

    return {
        "particle": particle,
        "energy_keV": energy_keV,
        "energy_text": energy_text,
        "suffix": suffix,
        "run_id": run_id,
    }

def parse_run_from_metrics_filename(fname: str):
    """
    returns dict with particle, energy_keV, energy_text, suffix, run_id
    Accepts:
      metrics_gamma_10keV.h5
      metrics_gamma_10keV_tcad.h5
    """
    m = re.match(
        r"^metrics_(?P<particle>.+)_(?P<energy>\d+(?:keV|MeV|GeV))(?:_(?P<suffix>.*))?\.h5$",
        fname,
        re.IGNORECASE
    )
    if not m:
        return None

    particle = m.group("particle")
    energy_text = m.group("energy")
    suffix = m.group("suffix")  # will be None if not present

    energy_keV, unit = parse_energy_to_keV(energy_text)
    if energy_keV is None:
        return None
    
    if suffix:
        run_id = f"{particle}_{energy_text}_{suffix}"
    else:
        run_id = f"{particle}_{energy_text}"

    return {
        "particle": particle,
        "energy_keV": energy_keV,
        "energy_text": energy_text,
        "suffix": suffix,
        "run_id": run_id,
    }

print("run parsing helpers ready.")


run parsing helpers ready.


In [18]:
# load all hdf5

metrics_dir = Path("./metrics")
files = sorted(metrics_dir.glob("metrics_*.h5"))

print("Found files:")
for f in files:
    print("  ", f.name)

all_runs = []

for file in files:
    info = parse_run_from_metrics_filename(file.name)

    if info is None:
        print(f"Skipping {file.name} (name not recognized)")
        continue

    with h5py.File(file, "r") as f:
        dset = f["clusters"]
        data_array = dset[:]
        columns = list(dset.attrs["columns"])

    df = pd.DataFrame(data_array, columns=columns)

    # add labels
    df["particle"] = info["particle"]
    df["energy_keV"] = info["energy_keV"]
    df["energy_text"] = info["energy_text"]
    df["suffix"] = info["suffix"]
    df["run"] = info["run_id"]
    df["source_file"] = file.name

    all_runs.append(df)

if not all_runs:
    raise ValueError("No valid metric files were loaded.")

data = pd.concat(all_runs, ignore_index=True)

print("\nLoaded dataframe shape:", data.shape)
print("Number of runs loaded:", data["run"].nunique())

print("\nAlpha 500keV runs loaded:")
for r in sorted(data["run"].unique()):
    if "alpha" in str(r).lower() and "500kev" in str(r).lower():
        print(" ", repr(r))


Found files:
   metrics_alpha_100MeV_1cm.h5
   metrics_alpha_100keV_1cm.h5
   metrics_alpha_10MeV_1cm.h5
   metrics_alpha_10keV_1cm.h5
   metrics_alpha_1MeV_1cm.h5
   metrics_alpha_200MeV_1cm.h5
   metrics_alpha_500MeV_1cm.h5
   metrics_alpha_500keV_0.01ts20e.h5
   metrics_alpha_500keV_0.02ts.h5
   metrics_alpha_500keV_0.02tstcad.h5
   metrics_alpha_500keV_0.07ts.h5
   metrics_alpha_500keV_0.07tstcad.h5
   metrics_alpha_500keV_0.10sp.h5
   metrics_alpha_500keV_0.12ts.h5
   metrics_alpha_500keV_0.12tstcad.h5
   metrics_alpha_500keV_0.17ts.h5
   metrics_alpha_500keV_0.17tstcad.h5
   metrics_alpha_500keV_0.22ts.h5
   metrics_alpha_500keV_0.22tstcad.h5
   metrics_alpha_500keV_0.25sp.h5
   metrics_alpha_500keV_0.27ts.h5
   metrics_alpha_500keV_0.27tstcad.h5
   metrics_alpha_500keV_0.32ts.h5
   metrics_alpha_500keV_0.32tstcad.h5
   metrics_alpha_500keV_0.37ts.h5
   metrics_alpha_500keV_0.37tstcad.h5
   metrics_alpha_500keV_0.42ts.h5
   metrics_alpha_500keV_0.42tstcad.h5
   metrics_alpha_500k

In [19]:
#charge to energy
# Q_cluster is total electrons per cluster (sum of |charge| per pixel)
# Silicon: ~3.6 eV per electron-hole pair
EV_PER_EH = 3.64

# Convert electrons -> keV and MeV
data["E_cluster_keV"] = data["Q_cluster"] * EV_PER_EH / 1000.0
# per-cluster "energy density" 
data["energy_density"] = data["E_cluster_keV"] / data["n_hits"].replace(0, np.nan)
#get rid of all Q_cluster
#data.drop(columns=["Q_cluster"], inplace=True)
# permanent master copy (IMPORTANT)
data_all = data.copy()

print("data_all shape:", data_all.shape)

data_all shape: (815884, 26)


In [20]:
# selection- can opt to only analyze spcific runs, can choose "all", "particles", "pairs", or "runs"
# creates:
#   data_all = full dataset (keep)
#   data     = filtered dataset (use for all later cells)

# use the already-created full dataset
data = data_all.copy()

# ------------------- CHOOSE ONE MODE -------------------


MODE = "runs"          

# option A: choose particles type
SELECT_PARTICLES = ["gamma"]     # e.g. ["alpha"] 

# option B: choose specific (particle, energy_keV) pairs
# remember: it has all been converted to keV so 1 MeV = 1000 keV, 10 MeV = 10000 keV, 1 GeV = 1_000_000 keV !!!
SELECT_PAIRS = [
    ("gamma", 10),
    ("alpha", 1000),
    ("proton", 10_000),
    ("electron", 100),
]

# option C: choose specific runs by filename 
# option C: choose specific runs by filename
SELECT_RUNS = [

    # alpha
    "alpha_100MeV_1cm",
    "alpha_100keV_1cm",
    "alpha_10MeV_1cm",
    "alpha_10keV_1cm",
    "alpha_1MeV_1cm",
    "alpha_200MeV_1cm",
    "alpha_500MeV_1cm",
    "alpha_50MeV_1cm",

    # electron
    "electron_100MeV_1cm",
    "electron_100keV_1cm",
    "electron_10MeV_1cm",
    "electron_1MeV_1cm",
    "electron_200MeV_1cm",
    "electron_200keV_1cm",
    "electron_500MeV_1cm",
    "electron_500keV_1cm",
    "electron_50keV_1cm",

    # gamma
    "gamma_100keV_1cm",
    "gamma_10MeV_1cm",
    "gamma_10keV_1cm",
    "gamma_1MeV_1cm",
    "gamma_500keV_1cm",

    # muon+
    "muon+_100keV_1cm",
    "muon+_10keV_1cm",
    "muon+_1MeV_1cm",
    "muon+_500keV_1cm",

    # muon-
    "muon-_100keV_1cm",
    "muon-_10keV_1cm",
    "muon-_1MeV_1cm",
    "muon-_500keV_1cm",

    # proton
    "proton_100MeV_1cm",
    "proton_100keV_1cm",
    "proton_10MeV_1cm",
    "proton_10keV_1cm",
    "proton_1MeV_1cm",
    "proton_200MeV_1cm",
    "proton_500MeV_1cm",
    "proton_50MeV_1cm",
]

# filter applied here
if MODE == "all":
    data = data_all.copy()

elif MODE == "particles":
    data = data_all[data_all["particle"].isin(SELECT_PARTICLES)].copy()

elif MODE == "pairs":
    mask = False
    for p, e in SELECT_PAIRS:
        mask = mask | ((data_all["particle"] == p) & (data_all["energy_keV"] == e))
    data = data_all[mask].copy()

elif MODE == "runs":
    data = data_all[data_all["run"].isin(SELECT_RUNS)].copy()

else:
    raise ValueError(f"Unknown MODE: {MODE}")

# summary
print("Selection MODE:", MODE)
print("Selected shape:", data.shape)
print("Selected particles:", sorted(data["particle"].unique()))
print("Selected runs:", sorted(data["run"].unique())[:15]) #can limit to only first x amount of runs


Selection MODE: runs
Selected shape: (342916, 26)
Selected particles: ['alpha', 'electron', 'gamma', 'muon+', 'muon-', 'proton']
Selected runs: ['alpha_100MeV_1cm', 'alpha_100keV_1cm', 'alpha_10MeV_1cm', 'alpha_10keV_1cm', 'alpha_1MeV_1cm', 'alpha_200MeV_1cm', 'alpha_500MeV_1cm', 'alpha_50MeV_1cm', 'electron_100MeV_1cm', 'electron_100keV_1cm', 'electron_10MeV_1cm', 'electron_1MeV_1cm', 'electron_200MeV_1cm', 'electron_200keV_1cm', 'electron_500MeV_1cm']


In [27]:
# Hit-event efficiency study
# Used sims that have been ran through clustering already (10k events, but can be scaled to 100k)
# Estimate how many incident particles are needed to get 10k hit events

TARGET_HIT_EVENTS = 10_000
# BUFFER = 0.10   # can add in an additional buffer too

hit_event_study = (
    data
    .groupby(["run", "particle", "energy_text"], dropna=False)
    .agg(
        events_with_pixel_hits=("event", "nunique"),
        clusters_total=("cluster_id", "count"),
    )
    .reset_index()
)

hit_event_study["n_events"] = TARGET_HIT_EVENTS

hit_event_study["events_without_pixel_hits"] = (
    hit_event_study["n_events"] - hit_event_study["events_with_pixel_hits"]
)

hit_event_study["hit_event_efficiency"] = (
    hit_event_study["events_with_pixel_hits"] / hit_event_study["n_events"]
)

hit_event_study["hit_event_percent"] = (
    100 * hit_event_study["hit_event_efficiency"]
)

hit_event_study["clusters_per_hit_event"] = (
    hit_event_study["clusters_total"] / hit_event_study["events_with_pixel_hits"]
)


hit_event_study["incident_needed_for_10k_hit_events"] = np.ceil(
   TARGET_HIT_EVENTS / hit_event_study["hit_event_efficiency"]
).astype(int)

#uncomment if deciding to add buffer calculation
#hit_event_study["incident_needed_with_10pct_buffer"] = np.ceil(
#    hit_event_study["incident_needed_for_10k_hit_events"] * (1 + BUFFER)
#).astype(int)

hit_event_study = hit_event_study.sort_values(
    ["particle", "energy_text"],
    ignore_index=True
)

hit_event_study = hit_event_study[
    [
        "run",
        "particle",
        "energy_text",
        "n_events",
        "events_with_pixel_hits",
        "events_without_pixel_hits",
        "hit_event_percent",
        "hit_event_efficiency",
        "incident_needed_for_10k_hit_events",
        "clusters_total",
        "clusters_per_hit_event",
    ]
]

hit_event_study

,run,particle,energy_text,n_events,events_with_pixel_hits,events_without_pixel_hits,hit_event_percent,hit_event_efficiency,incident_needed_for_10k_hit_events,clusters_total,clusters_per_hit_event
0,alpha_100MeV_1cm,alpha,100MeV,10000,9919,81,99.19,0.9919,10082,9936,1.001714
1,alpha_100keV_1cm,alpha,100keV,10000,9882,118,98.82,0.9882,10120,9882,1.000000
2,alpha_10MeV_1cm,alpha,10MeV,10000,9889,111,98.89,0.9889,10113,9947,1.005865
3,alpha_10keV_1cm,alpha,10keV,10000,9820,180,98.20,0.9820,10184,9820,1.000000
4,alpha_1MeV_1cm,alpha,1MeV,10000,9869,131,98.69,0.9869,10133,9869,1.000000
5,alpha_200MeV_1cm,alpha,200MeV,10000,9876,124,98.76,0.9876,10126,9938,1.006278
6,alpha_500MeV_1cm,alpha,500MeV,10000,9905,95,99.05,0.9905,10096,9987,1.008279
7,alpha_50MeV_1cm,alpha,50MeV,10000,9927,73,99.27,0.9927,10074,9931,1.000403
8,electron_100MeV_1cm,electron,100MeV,10000,9882,118,98.82,0.9882,10120,9979,1.009816
9,electron_100keV_1cm,electron,100keV,10000,9583,417,95.83,0.9583,10436,9772,1.019722


In [ ]:
out_csv = metrics_dir / "incident_particle_study.csv"
event_study.to_csv(out_csv, index=False)
print("Saved:", out_csv)